# Fine-tuning Phase

This notebook covers the second approach in my project: fine-tuning a language model on Austrian tax law data.

Unlike the inference notebook where I simply prompted GPT-4o-mini through an API, here I actually train a model to internalize the specific patterns, terminology, and structure of Austrian tax law. I use the Unsloth library, which is significantly faster and more memory-efficient than the standard Hugging Face `Trainer` used in a typical supervised fine-tuning (SFT) setup.

The notebook is structured in three parts: fine-tuning the model, testing it on a single question, and running bulk inference across all 644 dataset rows to produce a results CSV for BERTScore evaluation.

### Phase 1: Setup and Model Loading

Here I load a base model that has already been compressed into 4-bit precision. This compression (called quantization) is what allows an 8-billion parameter model to fit onto a single T4 GPU. Without it the model would require far more VRAM than a standard GPU provides.

I use Unsloth's `FastLanguageModel` instead of the standard HuggingFace `AutoModelForSequenceClassification` because it applies low-level kernel optimizations that reduce memory usage by roughly 70% and speed up training by 2x.

In [ ]:
!pip install "unsloth[colab-new]" "--upgrade"

import torch                           # PyTorch: the deep learning framework the model and training loop run on
import json                            # Used to open and parse my training_data.json file
from unsloth import FastLanguageModel  # Unsloth's optimized model loader, faster and more memory-efficient than standard HuggingFace
from datasets import Dataset           # Converts my JSON list into the HuggingFace Dataset format the SFTTrainer expects
from trl import SFTTrainer             # Supervised Fine-Tuning Trainer: manages the training loop automatically
from transformers import TrainingArguments  # Holds all training hyperparameters in one structured object
import os # Import os for environment variables

# Maximum token length per training example (question + answer combined).
# 2048 tokens is enough to cover even long Austrian tax law answers without truncating them.
max_seq_length = 2048

# I use Llama 3.1 8B pre-quantized to 4-bit precision via bitsandbytes.
# This is the same principle as loading a compressed model checkpoint:
# the weights are stored in 4-bit integers rather than 16-bit floats,
# which cuts VRAM usage roughly in half while keeping accuracy acceptable for fine-tuning.
model_id = "unsloth/Meta-Llama-3.1-8B-bnb-4bit"

# Workaround for HuggingFace TimeoutError as suggested by Unsloth
!pip install modelscope
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'

# Load the model and tokenizer together.
# The tokenizer converts raw German text into token IDs the model can process,
# and converts the model's output token IDs back into readable text after generation.
# This is the same AutoTokenizer pattern used in standard HuggingFace SFT, but wrapped by Unsloth.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_id,
    max_seq_length = max_seq_length,
    load_in_4bit = True,           # Keep the model in 4-bit format to stay within GPU memory limits
    token = "x",  # HuggingFace access token required to download gated models like Llama
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0

model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

Unsloth: Will load unsloth/Meta-Llama-3.1-8B-bnb-4bit as a legacy tokenizer.


### Phase 2: Applying LoRA (Low-Rank Adaptation)

Standard SFT updates every parameter in the model. With 8 billion parameters that is impossible on a single GPU. LoRA solves this by freezing all original model weights and inserting small trainable adapter matrices at specific layers. Only these adapters are updated during training.

This is conceptually similar to how `AutoModelForSequenceClassification` adds a new classification head on top of a frozen base model, except LoRA distributes the trainable capacity across multiple internal layers rather than adding it only at the output.

In [2]:
# Attach LoRA adapters to the frozen base model using Unsloth's PEFT integration
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,  # Rank: the inner dimension of the adapter matrices.
             # A higher rank gives the adapters more capacity to learn but uses more memory.
             # 16 is a balanced starting point for domain-specific fine-tuning.

    # The layers where adapters are inserted.
    # q/k/v/o_proj are the attention projection matrices (same layers targeted in most LoRA papers).
    # gate/up/down_proj are the feed-forward network layers inside each transformer block.
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],

    lora_alpha = 16,   # Scaling factor applied to the adapter outputs before adding them to the base model.
                       # Matching lora_alpha to r keeps the effective learning rate stable.

    lora_dropout = 0,  # No dropout during adapter training. Unsloth is optimized for this setting.

    bias = "none",     # Bias terms in the base model are not updated, only the adapter matrices.

    # Gradient checkpointing saves VRAM by not storing all intermediate activations during the forward pass.
    # Instead they are recomputed on demand during backpropagation.
    # The 'unsloth' mode is their most memory-efficient implementation of this technique.
    use_gradient_checkpointing = "unsloth",
)

Unsloth 2026.4.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


### Phase 3: Dataset Formatting

In standard SFT (as in the lecture notebook), the dataset provides `sentence1`, `sentence2`, and a `label`, and the tokenizer handles formatting automatically. For generative fine-tuning the setup is different: the model receives a single formatted string and learns to predict the answer tokens that follow the question tokens.

I use the Llama-3 prompt format with `### Frage:` and `### Antwort:` as delimiters. The `<|end_of_text|>` token at the end of each example tells the model where one training sample stops and the next begins, preventing it from generating text that bleeds across examples.

In [3]:
# Load my Austrian tax law training data from disk.
# The JSON file contains a list of question/answer dicts.
with open('training_data.json', 'r', encoding = 'utf-8') as f:
    json_data = json.load(f)

# Apply the Llama-3 prompt template to every example.
# The model learns the mapping: given '### Frage: ...' predict '### Antwort: ...'
# This is the generative equivalent of the tokenize_function in standard SFT.
def format_prompts(example):
    return {"text": f"### Frage: {example['question']}\n### Antwort: {example['answer']} <|end_of_text|>"}

# Convert the JSON list into a HuggingFace Dataset object (same class used in the lecture),
# then apply the formatting function across every row.
dataset = Dataset.from_list(json_data)
dataset = dataset.map(format_prompts)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

### Phase 4: Training Parameters

The `TrainingArguments` here follow the same structure as in the standard SFT lecture notebook, with a few differences driven by the GPU memory constraints of working with a large generative model instead of a smaller classification model.

In [4]:
training_args = TrainingArguments(
    per_device_train_batch_size = 2,  # 2 examples per GPU step. Smaller than the lecture's batch of 4
                                      # because the generative model is much larger and needs more VRAM per example.

    gradient_accumulation_steps = 4,  # Accumulate gradients over 4 steps before updating weights.
                                      # Effective batch size = 2 x 4 = 8, matching common SFT practice
                                      # without needing to fit 8 examples in VRAM simultaneously.

    warmup_steps = 5,                 # Ramp the learning rate up gradually over the first 5 steps
                                      # to avoid large destabilizing updates at the very start of training.
                                      # The lecture used warmup_steps=500 over a much longer run.

    max_steps = 60,                   # Run for 60 gradient update steps total.
                                      # Kept short to avoid overfitting on a limited tax law dataset.

    learning_rate = 2e-4,             # Standard learning rate for LoRA fine-tuning.
                                      # Higher than typical full fine-tuning rates because only the small
                                      # adapter matrices are being updated, not the full model.

    fp16 = not torch.cuda.is_bf16_supported(),  # Use 16-bit floating point for speed.
                                                # bf16 is preferred on newer GPUs; fp16 is the fallback.

    logging_steps = 1,                # Log the training loss after every single step.

    optim = "adamw_8bit",             # 8-bit AdamW optimizer: functionally identical to standard AdamW
                                      # but compresses optimizer states to 8-bit, saving significant VRAM.

    output_dir = "outputs",           # Folder where intermediate training checkpoints are written.
)

### Phase 5: The Actual Training

The `SFTTrainer` is the generative equivalent of the `Trainer` class used in the lecture notebook. It takes the model, the formatted dataset, and the training arguments and runs the full training loop: forward pass, loss calculation, backpropagation, and weight update.

The training ran for 60 steps. Below are 5 representative steps showing how the loss decreased as the model learned the Austrian tax law patterns:

| Step | Training Loss |
|------|---------------|
| 1    | 2.142186      |
| 15   | 0.924347      |
| 30   | 0.420398      |
| 45   | 0.301850      |
| 60   | 0.225400      |

In [5]:
# SFTTrainer is the generative counterpart to the standard Trainer from the lecture.
# dataset_text_field points it to the 'text' column created by format_prompts,
# which contains the fully formatted training strings.
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = training_args,
)

# Start the training run. The GPU computes the loss and updates the LoRA adapter weights each step.
trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=12):   0%|          | 0/100 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100 | Num Epochs = 5 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,2.142186
2,2.083776
3,2.111930
4,2.065987
5,1.793121
6,1.539401
7,1.269252
8,1.435261
9,1.292334
10,1.212924


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


TrainOutput(global_step=60, training_loss=0.742952527354161, metrics={'train_runtime': 160.0944, 'train_samples_per_second': 2.998, 'train_steps_per_second': 0.375, 'total_flos': 1531682083356672.0, 'train_loss': 0.742952527354161, 'epoch': 4.64})

### Phase 6: Saving the Result

I save only the LoRA adapter weights rather than the full model. The adapters hold all the Austrian tax law knowledge the model learned and are a small fraction of the full model size. They can be reloaded later by merging them back into the original base model.

In [6]:
# Save the trained LoRA adapter weights to a local folder
model.save_pretrained("llama3_1_tax_lora")

# Save the tokenizer alongside so both can be reloaded consistently for inference
tokenizer.save_pretrained("llama3_1_tax_lora")

print("Training complete! Model saved in 'llama3_1_tax_lora'")

Unsloth: Restored added_tokens_decoder metadata in llama3_1_tax_lora/tokenizer_config.json.


Training complete! Model saved in 'llama3_1_tax_lora'


# (Bulk Inference) Production Phase

The single-question test confirmed the model works. Now I run it across all 644 rows of my dataset to produce `fine_tuned_results.csv` for BERTScore evaluation. To handle this volume efficiently I use batching, which sends multiple questions to the GPU simultaneously rather than one at a time.

### Phase 1: Environment and Pipeline Setup

I configure the pipeline to distribute the model across all available GPUs automatically using `device_map='auto'`, and to use `bfloat16` precision which reduces memory usage while maintaining enough numerical accuracy for generation.

In [7]:
import pandas as pd
import torch
from transformers import pipeline
from tqdm import tqdm  # Wraps the loop to display a live progress bar in the console
import csv

# Load the full 644-row test dataset
test_df = pd.read_csv('dataset_clean.csv')

# Initialize the generation pipeline.
# device_map='auto' spreads the model layers across all available GPUs automatically.
# dtype=torch.bfloat16 reduces memory per activation while keeping generation quality high.
generator = pipeline(
    "text-generation",
    model = model,
    tokenizer = tokenizer,
    device_map = "auto",
    dtype = torch.bfloat16
)

### Phase 2: Batching Strategy

Processing questions one at a time leaves the GPU underutilized between calls. Batching sends a group of questions in a single forward pass, keeping the GPU fully occupied and reducing total runtime significantly. A batch size of 8 means 8 questions are processed simultaneously per step.

In [8]:
batch_size = 8  # Number of questions processed simultaneously per GPU forward pass
results = []    # Collects id/answer dicts as the loop progresses

print(f"Starting inference for {len(test_df)} rows...")

# Iterate through the dataset in steps of 8 (indices 0, 8, 16, 24 and so on)
for i in tqdm(range(0, len(test_df), batch_size)):

    # Slice out the current batch of up to 8 rows
    batch = test_df.iloc[i:i+batch_size]

    # Format every prompt in the batch using the same structure used during training
    prompts = [f"### Frage:\n{row['prompt']}\n\n### Antwort:\n" for _, row in batch.iterrows()]

    # Run generation on the current batch.
    # max_new_tokens=150 gives enough room for a full legal citation and explanation.
    # do_sample=False: greedy decoding, always picks the most likely next token.
    # pad_token_id prevents warnings when sequences in the batch have different lengths.
    outputs = generator(
        prompts,
        max_new_tokens = 150,
        do_sample = False,
        pad_token_id = tokenizer.eos_token_id,
        batch_size = batch_size
    )

    # Loop through the 8 outputs from this batch
    for j, output in enumerate(outputs):
        generated_text = output[0]['generated_text']

        # Split at the answer marker and keep only what comes after it
        clean_answer = generated_text.split("### Antwort:\n")[-1].strip()

        # Store the answer with its original question ID to enable correct merging during evaluation
        results.append({
            "id": batch.iloc[j]['id'],
            "answer": clean_answer
        })


Starting inference for 643 rows...


100%|██████████| 81/81 [00:00<00:00, 2668.43it/s]


### Phase 3: The Generation Engine

The generator processes all 8 prompts in the current batch in a single forward pass. Setting `do_sample=False` uses greedy decoding, meaning the model always picks the highest-probability next token. This makes the output deterministic and consistent, which is important for reproducible evaluation.

### Phase 4: Data Cleaning and Extraction

Each output contains the full prompt plus the generated answer as one string. I strip the prompt portion from every output so only the clean answer is stored, then map it back to its original question ID so the results can be correctly merged with ground truth during BERTScore evaluation.

### Phase 5: Secure Export

I save the results using `csv.QUOTE_ALL`, which wraps every cell in double quotes. Austrian legal text frequently contains commas inside law citations such as `§ 2, Abs. 1`. Without quoting those commas would be misread as column separators, corrupting the CSV structure.

In [11]:
output_file = 'fine_tuned_results.csv'
results_df = pd.DataFrame(results)

# QUOTE_ALL wraps every cell in double quotes to protect commas inside legal citations
results_df.to_csv(output_file, index = False, quoting = csv.QUOTE_ALL, encoding = 'utf-8')

print(f"\nSuccess! Inference complete. File saved as: {output_file}")


Success! Inference complete. File saved as: fine_tuned_results.csv
